# Band Structure of Twisted MoS2 Bilayers.

> **Kaihui Liu, Liming Zhang, Ting Cao, Chenhao Jin, Diana Qiu, Qin Zhou, Alex Zettl, Peidong Yang, Steve G. Louie & Feng Wang**
> Evolution of interlayer coupling in twisted molybdenum disulfide bilayers. Nature Communications, 5, 4966. 2014.
> [https://doi.org/10.1038/ncomms5966](https://doi.org/10.1038/ncomms5966)

Calculate the band structure and the band gaps of the twisted MoS2 bilayers created in the
[structure notebook](interface_bilayer_twisted_commensurate_lattices_molybdenum_disulfide.ipynb),
using Quantum ESPRESSO and the band structure + density of states workflow from Standata.

The article's result is that the **indirect** bandgap of a MoS2 bilayer is set by the interlayer
distance alone: registered AA/AB stacking sits closer together and has a markedly smaller indirect
gap, every twisted angle sits further apart and lands on the same higher value, and the horizontal
alignment of the two layers plays no part beyond setting that distance. The K-valley **direct** gap
barely moves throughout.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_bilayer_twisted_commensurate_lattices_molybdenum_disulfide.ipynb), which saves them to the `uploads` folder under the names used in cell 1.2 below.
1. Set the materials and the calculation parameters in cells 1.2 and 1.3 (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the materials, workflow, compute resources and jobs.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the materials by name from the `uploads` folder and save them to the platform.
1. Configure the workflow: select the application, load the band structure + DOS workflow from Standata, and set the model and computational parameters for each material.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Create one job per material.
1. Submit the jobs and wait for them to complete.
1. Retrieve results: band structures, band gaps, and the comparison with the article.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and jobs

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used

# 3. Material parameters
FOLDER = "./uploads"
# Name of each structure saved by the structure notebook, its twist angle, and the k-grid to
# compute it on.
MATERIALS = {
    "MoS2 bilayer 21.8deg d6.5": {"angle": 21.8, "kgrid": [6, 6, 1]},
    # "MoS2 bilayer AB1 d6.1": {"angle": 60.0, "kgrid": [12, 12, 1]},
    # "MoS2 bilayer AB1 d6.5": {"angle": 60.0, "kgrid": [12, 12, 1]},
    # "MoS2 bilayer AA3 d6.8": {"angle": 0.0, "kgrid": [12, 12, 1]},
    # "MoS2 bilayer 13.2deg d6.5": {"angle": 13.2, "kgrid": [3, 3, 1]},
    # "MoS2 bilayer 38.2deg d6.5": {"angle": 38.2, "kgrid": [6, 6, 1]},
    # "MoS2 bilayer 46.8deg d6.5": {"angle": 46.8, "kgrid": [3, 3, 1]},
}

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "band_structure_dos.json"
MY_WORKFLOW_NAME = "Band Structure + DOS"
APPLICATION_NAME = "espresso"
MODEL_SUBTYPE = "lda"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set the DFT parameters

In [ ]:
# The publication used LDA with norm-conserving pseudopotentials; these are LDA ultrasoft, so
# expect somewhat different absolute values.
PSEUDOPOTENTIAL_TYPE = "us"
FUNCTIONAL = "pz"

# Ultrasoft pseudopotentials need a charge-density cutoff 8-12x the wavefunction one
ECUTWFC = 40
ECUTRHO = 8 * ECUTWFC

KPATH = [
    {"point": "Γ", "steps": 10},
    {"point": "M", "steps": 10},
    {"point": "K", "steps": 10},
    {"point": "Γ", "steps": 1},
]

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder

The structures, their geometry and their lattice type all come from the structure notebook; this one
only loads them by name.

In [ ]:
import numpy as np
from mat3ra.notebooks_utils.material import load_material_from_folder

materials = {}
for name in MATERIALS:
    material = load_material_from_folder(FOLDER, name, verbose=False)
    if material is None:
        raise ValueError(f"No material named '{name}' in '{FOLDER}'. Run the structure notebook "
                         f"first, or correct MATERIALS.")
    materials[name] = material
    print(f"{name}: {len(material.basis.elements.ids)} atoms, a = {material.lattice.a:.3f} Å, "
          f"lattice {material.lattice.type}")

### 3.2. Preview the materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": material, "title": name} for name, material in materials.items()],
                    viewer="wave")

### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for name, material in materials.items():
    material.basis.set_labels_from_list([])
    saved_materials[name] = Material.create(get_or_create_material(client, material, ACCOUNT_ID))

## 4. Configure the workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Configure the workflow

One workflow per material, each using the k-grid set alongside its name in 1.2.

The structures are not relaxed: the interlayer distances are the article's relaxed results, used
here as inputs.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode import ModelFactory
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider

from copy import deepcopy

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
path_context = PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data()

workflows = {}
for name, settings in MATERIALS.items():
    workflow = Workflow.create(deepcopy(workflow_config))
    workflow.name = f"{MY_WORKFLOW_NAME} {name}"
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model
    grid_context = PointsGridDataProvider(material=materials[name], dimensions=settings["kgrid"], isEdited=True).get_context_item_data()

    for unit_name, contexts in [("pw_scf", [grid_context, cutoffs_context]),
                                ("pw_nscf", [grid_context, cutoffs_context]),
                                ("pw_bands", [path_context, cutoffs_context])]:
        unit = subworkflow.get_unit_by_name(name=unit_name)
        for context in contexts:
            unit.add_context(context)
        subworkflow.set_unit(unit)
    workflows[name] = workflow
    print(f"{name}: k-grid {settings['kgrid']}")

### 4.3. Preview the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

visualize_workflow(workflows[next(iter(MATERIALS))])

### 4.4. Save the workflows to the collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

for name, workflow in workflows.items():
    saved = Workflow.create(get_or_create_workflow(client, workflow, ACCOUNT_ID))
    print(f"{name}: workflow {saved.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create one job per material

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.utils.namespace import dict_to_namespace_recursive

job_ids = {}
for name in MATERIALS:
    job_response = create_job(
        api_client=client,
        materials=[saved_materials[name]],
        workflow=workflows[name],
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {name} {timestamp}",
        compute=compute.to_dict(),
    )
    job_ids[name] = dict_to_namespace_recursive(job_response)._id
    print(f"✅ {name}: job {job_ids[name]}")

## 7. Submit the jobs and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if not job_ids:
    raise ValueError("No jobs were created - see the output of the previous cell.")

submit_jobs(client.jobs, list(job_ids.values()))
print(f"✅ Submitted {len(job_ids)} job(s).")

await wait_for_jobs_to_finish_async(client.jobs, list(job_ids.values()), poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. Band structures

The path is Γ–M–K–Γ of the cell being computed. For a supercell the bands are folded onto that
smaller Brillouin zone, so the same label is not the same point in the monolayer's zone; the band
gaps in 8.2 are read from the k-mesh instead and are unaffected by the folding.

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

for name, job_id in job_ids.items():
    property_data = get_properties_for_job(client, job_id, property_name="band_structure")
    visualize_properties(property_data, title=f"Band Structure: {name}",
                         extra_config={"material": materials[name].to_dict()})

### 8.2. Band gaps

`direct` is the smallest gap at a single k-point, which for MoS2 is the K-valley transition, and
`indirect` is the fundamental gap between the valence maximum and the conduction minimum wherever
they fall.

In [ ]:
from mat3ra.prode import PropertyName

gaps = {}
for name, job_id in job_ids.items():
    properties = client.properties.get_for_job(job_id, PropertyName.non_scalar.band_gaps.value)
    if not properties:
        raise ValueError(f"Job {job_ids[name]} for '{name}' produced no band gaps.")
    gaps[name] = {value["type"]: value for value in properties[0]["values"]}

print(f"{'material':<28} {'indirect':>10} {'direct':>10}  units")
for name in MATERIALS:
    indirect, direct = gaps[name]["indirect"], gaps[name]["direct"]
    print(f"{name:<28} {indirect['value']:>10.3f} {direct['value']:>10.3f}  {indirect.get('units')}")

## 9. Compare with the article

Fig. 4b of the article plots both gaps against twist angle.

In [ ]:
from mat3ra.notebooks_utils.plot import plot_series

# Read off Fig. 4b of the article: the K-valley direct gap is near enough constant, and the
# indirect gap is smaller for the registered stackings at 0 and 60 degrees than for the twists.
ARTICLE_INDIRECT_GAP = {0.0: 1.60, 13.2: 1.47, 21.8: 1.47, 38.2: 1.47, 46.8: 1.47, 60.0: 1.27}
ARTICLE_DIRECT_GAP = 1.80

series = sorted(
    (
        {
            "angle": MATERIALS[name]["angle"],
            "indirect": gaps[name]["indirect"]["value"],
            "direct": gaps[name]["direct"]["value"],
            "article indirect": ARTICLE_INDIRECT_GAP.get(MATERIALS[name]["angle"], float("nan")),
        }
        for name in gaps
    ),
    key=lambda item: item["angle"],
)

print(f"{'angle':>7} {'indirect':>10} {'direct':>10} {'article indirect':>18}")
for item in series:
    print(f"{item['angle']:>7} {item['indirect']:>10.3f} {item['direct']:>10.3f} "
          f"{item['article indirect']:>18.3f}")

plot_series(series=series, x_key="angle", y_key="indirect",
            xlabel="Twist angle (degrees)", ylabel="Indirect band gap (eV)",
            title="Indirect band gap vs twist angle")
plot_series(series=series, x_key="angle", y_key="direct",
            xlabel="Twist angle (degrees)", ylabel="K-valley direct band gap (eV)",
            title="Direct band gap vs twist angle")